# Metabolic "Orphan" Predicting Immune Evasion

**Research Question 1:** Can metabolic "orphan" interactions predict immune evasion across tumor types?

This notebook systematically cross-references our computationally predicted, literature-sparse ("Tier 2/3") metabolic target pairs against immune cell populations (such as B-cells, macrophages, and dendritic cells) in our cancer datasets. 

By identifying uncharacterized metabolic ligands that are heavily upregulated in these immune populations, we can potentially discover novel, unpatented immune-checkpoints driven by the metabolic microenvironment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import sys

# Define workspace
workspace_dir = '../'
output_dir = os.path.join(workspace_dir, 'output')
sys.path.append(output_dir)

# We use the parse_md_tables script to extract previously computed DE tables from HTML reports
from parse_md_tables import TableExtractor

print("Loading comprehensive target pairs...")
pairs_df = pd.read_csv(os.path.join(output_dir, 'human_database_merge_unique_metab_target_pairs_with_HMDB_Info.csv'))

# Explode Target column just in case it has commas
pairs_df['Target'] = pairs_df['Target'].astype(str).str.split(r'[,;]')
pairs_df = pairs_df.explode('Target')
pairs_df['Target'] = pairs_df['Target'].str.strip()

print(f"Total unique pairs loaded: {len(pairs_df)}")


## 1. Isolate the "Orphan" (Tier 2 & 3 Pairs)

We define "Orphan" pairs as those that are supported by 2-3 databases but lack heavy, redundant validation (which would make them Tier 1).

In [ ]:
def calc_tier(db_str):
    if pd.isna(db_str): return 'Tier 3 (Low)'
    count = len(str(db_str).split(','))
    if count >= 4: return 'Tier 1 (High)'
    elif count >= 2: return 'Tier 2 (Medium)'
    else: return 'Tier 3 (Low)'

pairs_df['Pair_Confidence_Tier'] = pairs_df['database'].apply(calc_tier)

orphan_metabolic_pairs = pairs_df[pairs_df['Pair_Confidence_Tier'].isin(['Tier 2 (Medium)', 'Tier 3 (Low)'])]
print(f"Orphan pairs identified: {len(orphan_metabolic_pairs)}")


## 2. Extract Highly Enriched Immune Targets

We parse the existing `cancer_*.html` reports to extract the genome-wide Differential Expression (DE) tables. We specifically filter for targets highly upregulated (`Log2FC > 1.0`, `padj < 0.05`) in immune clusters like B cells, Macrophages, and Dendritic cells.

In [ ]:
html_files = glob.glob(os.path.join(output_dir, '*.html'))
immune_enriched_genes = []

# Rigorous site-specific calculation using the scRNA-seq AnnData object
h5ad_file = globals().get('h5ad_path', None)
calculated_de = False
immune_de_df = None

if h5ad_file and os.path.exists(h5ad_file):
    try:
        import scanpy as sc
        print(f"Loading scRNA-seq data to compute site-specific immune enrichment: {os.path.basename(h5ad_file)}")
        adata = sc.read_h5ad(h5ad_file)
        
        primary_list = globals().get('PRIMARY_TISSUES', ['breast', 'mammary gland'])
        adata.obs['site'] = np.where(adata.obs['tissue_general'].isin(primary_list), primary_list[0].capitalize() if primary_list else 'Primary', adata.obs['tissue_general'])
        
        meta_tissues = [t for t in adata.obs['tissue_general'].unique() if t not in primary_list and pd.notnull(t)]
        top_3_meta = list(adata.obs[adata.obs['tissue_general'].isin(meta_tissues)]['tissue_general'].value_counts().head(3).index)
        valid_sites = [primary_list[0].capitalize() if primary_list else 'Primary'] + top_3_meta
        print(f"Target Sites for Analysis: {valid_sites}")
        
        adata_filtered = adata[adata.obs['site'].isin(valid_sites)].copy()
        
        immune_pattern = 'B cell|Macrophage|mononuclear phagocyte|dendritic cell|T cell'
        immune_mask = adata_filtered.obs['cell_type'].astype(str).str.contains(immune_pattern, case=False, na=False)
        
        site_de_results = []
        for site in valid_sites:
            adata_site = adata_filtered[adata_filtered.obs['site'] == site].copy()
            if len(adata_site) < 20: continue
            
            site_immune_mask = adata_site.obs['cell_type'].astype(str).str.contains(immune_pattern, case=False, na=False)
            if site_immune_mask.sum() < 5 or (~site_immune_mask).sum() < 5: continue
            
            adata_site.obs['is_immune'] = np.where(site_immune_mask, 'Immune', 'Other')
            sc.tl.rank_genes_groups(adata_site, groupby='is_immune', groups=['Immune'], reference='Other', method='t-test')
            
            result = adata_site.uns['rank_genes_groups']
            site_de = pd.DataFrame({
                'names': result['names']['Immune'],
                'logfoldchanges': result['logfoldchanges']['Immune'],
                'pvals_adj': result['pvals_adj']['Immune'],
                'group': ['Immune'] * len(result['names']['Immune']),
                'Cancer': [site] * len(result['names']['Immune'])
            })
            site_de = site_de[(site_de['logfoldchanges'] > 0) & (site_de['pvals_adj'] < 0.05)]
            site_de_results.append(site_de)
            
        if site_de_results:
            immune_de_df = pd.concat(site_de_results, ignore_index=True)
            print(f"Successfully calculated site-specific immune enrichment for: {list(immune_de_df['Cancer'].unique())}")
            calculated_de = True
    except Exception as e:
        print(f"Warning: Failed site-specific computation: {e}. Falling back to HTML table parser.")

if not calculated_de:
    print("Parsing DE tables from HTMLs...")
    for f in html_files:
        cancer_name = globals().get('CANCER_TYPE_NAME', os.path.basename(f)).replace('cancer_', '').split('_')[0]
        cancer_name = cancer_name.replace('_results', '').replace('_cancer', '')
        try:
            with open(f, 'r', encoding='utf-8') as inf:
                content = inf.read()
            extractor = TableExtractor()
            extractor.feed(content)
            
            de_table = None
            for tbl in extractor.tables:
                if not tbl: continue
                header = tbl[0]
                if 'logfoldchanges' in header and 'names' in header and 'group' in header:
                    de_table = pd.DataFrame(tbl[1:], columns=header)
                    break
            
            if de_table is not None:
                de_table['logfoldchanges'] = pd.to_numeric(de_table['logfoldchanges'], errors='coerce')
                de_table['pvals_adj'] = pd.to_numeric(de_table['pvals_adj'], errors='coerce')
                
                immune_mask = de_table['group'].str.contains('B cell|Macrophage|mononuclear phagocyte|dendritic cell|T cell', case=False, na=False)
                sig_mask = (de_table['logfoldchanges'] > 1.0) & (de_table['pvals_adj'] < 0.05)
                filtered = de_table[immune_mask & sig_mask].copy()
                filtered['Cancer'] = cancer_name
                immune_enriched_genes.append(filtered)
        except Exception as e:
            print(f"Error parsing {f}: {e}")

    if not immune_enriched_genes:
        print("❌ CRITICAL ERROR: No immune cell DE tables found.")
        raise ValueError("Pipeline failed to extract real Differential Expression data. DO NOT mock this data.")
    else:
        immune_de_df = pd.concat(immune_enriched_genes, ignore_index=True)

print(f"Found {len(immune_de_df['names'].unique())} unique highly enriched immune target genes across sites/cancers.")


In [ ]:
# Build orphan_metabolic_immune_df by merging immune DE results with orphan pairs
target_genes = set(orphan_metabolic_pairs['Target'].dropna().unique())

orphan_metabolic_immune_df = immune_de_df[immune_de_df['names'].isin(target_genes)].copy()
orphan_metabolic_immune_df = orphan_metabolic_immune_df.merge(
    orphan_metabolic_pairs[['Target', 'Metabolite_Name']].drop_duplicates(),
    left_on='names', right_on='Target', how='inner'
).drop(columns=['Target'])
orphan_metabolic_immune_df = orphan_metabolic_immune_df.rename(columns={'names': 'Target', 'group': 'cell_type'})
orphan_metabolic_immune_df = orphan_metabolic_immune_df[orphan_metabolic_immune_df['logfoldchanges'] > 0].copy()
print(f"Orphan metabolic immune interactions: {len(orphan_metabolic_immune_df)}")

## 3. Map Orphan Metabolites to Immune Targets

We cross-reference our immune DE genes with the Orphan pairs, and visualize the top hits.

In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_orphan_metabolic_multipanel(df, top_n=30, figsize_per_panel=(10, 10), order=None):
    """
    Multi-panel bar plot: each panel = one tumor site (Cancer),
    y-axis = Metabolite → Target interaction, x-axis = logfoldchanges.
    """
    sites = df['Cancer'].unique()
    ncols = min(3, len(sites))
    nrows = math.ceil(len(sites) / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows))
    axes = np.array(axes).flatten()

    for i, site in enumerate(sites):
        ax = axes[i]
        subset = df[df['Cancer'] == site].copy()
        subset['Interaction'] = subset['Metabolite_Name'] + ' → ' + subset['Target']
        top_hits = subset.sort_values('logfoldchanges', ascending=False).drop_duplicates('Interaction').head(top_n)

        if order is not None:
            sns.barplot(data=subset, x='logfoldchanges', y='Interaction', color='steelblue', ax=ax, order=order)
        else:
            sns.barplot(data=top_hits, x='logfoldchanges', y='Interaction', color='steelblue', ax=ax)

        ax.set_title(site, fontsize=13, fontweight='bold')
        ax.set_xlabel('Log2 Fold Change (Immune vs Other)')
        ax.set_ylabel('Metabolite → Target' if i % ncols == 0 else '')
        ax.grid(True, linestyle='--', alpha=0.5, axis='x')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        'Orphan Metabolic Interactions in Immune Cells\nacross Tumor Sites',
        fontsize=15, y=1.01
    )
    plt.tight_layout()
    plt.show()

if len(orphan_metabolic_immune_df) == 0:
    print("⚠️ No orphan metabolic immune interactions found, skipping plot.")
else:
    plot_orphan_metabolic_multipanel(orphan_metabolic_immune_df)

### Overlapping Interactions Across Tumor Sites
Filtering to show only interactions (Metabolite → Target) that are conserved across multiple tumor sites.


In [ ]:
# Identify overlapping interactions across tumor sites
df_with_interaction = orphan_metabolic_immune_df.copy()
df_with_interaction["Interaction"] = df_with_interaction["Metabolite_Name"] + " → " + df_with_interaction["Target"]

# Count how many unique tumor sites each interaction appears in
site_counts = df_with_interaction.groupby("Interaction")["Cancer"].nunique()

# Filter for interactions present in more than 1 site
overlapping_interactions = site_counts[site_counts > 1].index

if len(overlapping_interactions) > 0:
    print(f"Found {len(overlapping_interactions)} interactions overlapping across multiple tumor sites.")
    overlapping_df = df_with_interaction[df_with_interaction["Interaction"].isin(overlapping_interactions)]
    # The plot function creates "Interaction" column again internally, which is fine.
    primary_site = overlapping_df["Cancer"].unique()[0]
    primary_order = overlapping_df[overlapping_df["Cancer"] == primary_site].sort_values("logfoldchanges", ascending=False)["Interaction"].tolist()
    other_interactions = [i for i in overlapping_interactions if i not in primary_order]
    common_order = primary_order + other_interactions
    common_order = common_order[:30] # Keep top 30 based on primary
    plot_orphan_metabolic_multipanel(overlapping_df, top_n=30, order=common_order)
else:
    print("No overlapping orphan metabolic interactions found across multiple tumor sites.")


In [ ]:
# Save the comprehensive list for manual review
output_csv = os.path.join(output_dir, 'immune_evasion_orphan_metabolic_candidates.csv')
orphan_metabolic_immune_df.to_csv(output_csv, index=False)
print(f"Saved highly confident orphan candidates to {output_csv}")

# Display top 10
orphan_metabolic_immune_df[['Cancer', 'cell_type', 'Target', 'logfoldchanges', 'Metabolite_Name']]